## Key Stats Comparison

**Location:** 1303 Woodbury Ave, Portsmouth, NH  
**Coordinates:** latitude 43.0859778, longitude -70.78677

**Objective:** Compare key demographic and traffic stats against Washville median benchmarks

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='.env')

placer_api_key = os.getenv('PLACER_API_KEY')

if not placer_api_key:
    raise ValueError("PLACER_API_KEY not found in .env file")

print("✓ API keys loaded successfully")

In [ ]:
import requests
import time
from datetime import datetime, timedelta
from pydantic import BaseModel
from typing import Optional, List

print("✓ Imports loaded")

In [ ]:
# Pydantic models for POI search
class CategoryInfo(BaseModel):
    category: str
    group: str
    subCategory: str

class Address(BaseModel):
    city: str
    state: str
    countryCode: str
    streetName: str
    formattedAddress: str
    shortFormattedAddress: str
    zipCode: str

class RegionDetail(BaseModel):
    code: str
    name: str

class Regions(BaseModel):
    dma: RegionDetail
    state: RegionDetail
    cbsa: RegionDetail

class Venue(BaseModel):
    entityId: str
    entityType: str
    name: str
    categoryInfo: CategoryInfo
    address: Address
    isFlagged: bool
    regions: Regions
    apiId: str
    placerUrl: str
    storeId: Optional[str] = None
    isPermitted: bool

class PlacerResponse(BaseModel):
    data: List[Venue]
    requestId: str

print("✓ Pydantic models defined")

## Step 1: Find Reference POI

In [ ]:
# Target POI coordinates
target_lat = 43.0859778
target_lng = -70.78677

# Search for ANY venue near the target location
target_search_url = "https://papi.placer.ai/v1/poi"
target_params = {
    "lat": target_lat,
    "lng": target_lng,
    "radius": 0.5,
    "entityType": "venue",
    "limit": 20
}

headers = {
    "accept": "application/json",
    "x-api-key": placer_api_key
}

print(f"Searching for businesses near target location...\n")

response = requests.get(target_search_url, params=target_params, headers=headers)

reference_poi_id = None
reference_poi_name = None

if response.status_code == 200:
    result = response.json()
    
    if result.get('data'):
        parsed_result = PlacerResponse.model_validate_json(response.text)
        
        print(f"Found {len(parsed_result.data)} nearby businesses\n")
        
        if parsed_result.data:
            reference_poi_id = parsed_result.data[0].apiId
            reference_poi_name = parsed_result.data[0].name
            print(f"✓ Using as reference POI: {reference_poi_name}")
            print(f"  API ID: {reference_poi_id}\n")
else:
    print(f"Error: {response.text}")

## Step 2: Pull Demographics for 15 Min Drive Time

Get car parc, median income, and average age from demographics API.

In [ ]:
# Date range
end_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=150)).strftime("%Y-%m-%d")

# Demographics API endpoint
demographics_url = "https://papi.placer.ai/v1/reports/trade-area-demographics"

headers_demographics = {
    "accept": "application/json",
    "content-type": "application/json",
    "x-api-key": placer_api_key
}

# Use 15 minute drive time for key stats
drive_time = 15

payload = {
    "method": "driveTime",
    "benchmarkScope": "nationwide",
    "allocationType": "weightedCentroid",
    "trafficVolPct": 70,
    "withinRadius": drive_time,
    "ringRadius": 3,
    "dataset": "sti_popstats",
    "startDate": start_date,
    "endDate": end_date,
    "apiId": reference_poi_id,
    "driveTime": drive_time,
    "template": "default"
}

print(f"Fetching demographics for {drive_time} min drive time...\n")

# Retry loop for IN_PROGRESS responses
max_attempts = 15
demographics_data = None

for attempt in range(max_attempts):
    response = requests.post(demographics_url, json=payload, headers=headers_demographics)
    
    if response.status_code not in [200, 202]:
        print(f"✗ HTTP {response.status_code}")
        break
    
    if "IN_PROGRESS" in response.text or response.status_code == 202:
        if attempt == 0:
            print("Processing", end="")
        print(".", end="", flush=True)
        time.sleep(2)
        continue
    
    # Success - extract data
    try:
        result = response.json()
        demographics_data = result.get('data', {})
        print(" ✓")
        break
    except Exception as e:
        print(f" ✗ Error: {str(e)[:50]}")
        break

if not demographics_data:
    print("\n⚠️  Failed to retrieve demographics data")

## Step 3: Extract Key Stats

In [ ]:
if demographics_data:
    # Extract key metrics
    overview = demographics_data.get('Overview', {})
    vehicles = demographics_data.get('Vehicles per Household', {})
    
    # Car Parc
    car_parc = vehicles.get('Total Number of Vehicles', {}).get('value', 0)
    
    # Median Income
    median_income = overview.get('Household Median Income', {}).get('value', 0)
    
    # Average Age
    average_age = overview.get('Average Age', {}).get('value', 0)
    
    # Population
    population = overview.get('Population', {}).get('value', 0)
    
    # Households
    households = overview.get('Households', {}).get('value', 0)
    
    print("\nKEY STATS EXTRACTED")
    print("=" * 60)
    print(f"\nCar Parc (15 min):      {car_parc:>15,.0f}")
    print(f"Median Income:          ${median_income:>14,.0f}")
    print(f"Average Age:            {average_age:>15.1f}")
    print(f"Population:             {population:>15,.0f}")
    print(f"Households:             {households:>15,.0f}")
    print("\n" + "=" * 60)
else:
    print("\n⚠️  No demographics data available")
    car_parc = 0
    median_income = 0
    average_age = 0

## Step 4: Compare to Washville Median

**Note:** Car Counts would come from traffic/Inrix data (not implemented yet)

In [ ]:
# Washville Median benchmarks (from historical data)
washville_median = {
    'Car Counts': 23_000,  # << Placeholder - would come from traffic/Inrix
    'Car Parc': 96_000,
    'Median Income': 83_000,
    'Average Age': 41.0
}

# Current site stats
portsmouth_stats = {
    'Car Counts': 21_000,  # << Placeholder - would come from traffic/Inrix  
    'Car Parc': int(car_parc),
    'Median Income': int(median_income),
    'Average Age': round(average_age, 1)
}

# Calculate variances
print("\nKEY STATS COMPARISON")
print("=" * 80)
print(f"\n{'Metric':<20} {'Portsmouth':>20} {'Washville Median':>20}")
print("-" * 80)

for metric in ['Car Counts', 'Car Parc', 'Median Income', 'Average Age']:
    portsmouth_val = portsmouth_stats[metric]
    median_val = washville_median[metric]
    
    if median_val > 0:
        variance = ((portsmouth_val - median_val) / median_val * 100)
        variance_str = f"{variance:+.1f}%"
    else:
        variance_str = "N/A"
    
    # Format values
    if metric in ['Car Counts', 'Car Parc']:
        port_str = f"{portsmouth_val:,}"
        med_str = f"{median_val:,}"
    elif metric == 'Median Income':
        port_str = f"${portsmouth_val:,}"
        med_str = f"${median_val:,}"
    else:  # Average Age
        port_str = f"{portsmouth_val:.1f}"
        med_str = f"{median_val:.1f}"
    
    print(f"{metric:<20} {port_str:>20} {med_str:>20}")

print("-" * 80)
print("\nNote: Car Counts would be pulled from traffic/Inrix data (placeholder value shown)")
print("=" * 80)